In [1]:
"""
时间合成分析脚本 - 潜热通量、表面风速和比湿差
基于降水事件检测结果进行时间维度的合成分析
"""
import xarray as xr
import numpy as np
import os
import json
from datetime import datetime
from pathlib import Path
import sys

# 添加 wave_tools 到路径
WAVE_TOOLS_PATH = Path("/work/mh1498/m301257/wave_tools")
sys.path.insert(0, str(WAVE_TOOLS_PATH.parent))
from wave_tools import CCKWFilter

# ============================================================================
# 配置参数
# ============================================================================
EXPERIMENTS = ['CNTL', 'P4K', '4CO2']
COMPOSITE_DIR = '../composite_data/'
OUTPUT_DIR = '/work/mh1498/m301257/composite_data/'

# 创建输出目录
os.makedirs(OUTPUT_DIR, exist_ok=True)

print("="*80)
print("🔄 时间合成分析 - LHF, Surface Wind, Humidity Difference")
print("="*80)
print(f"📅 开始时间: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print(f"📁 输出目录: {OUTPUT_DIR}")
print("="*80)

🔄 时间合成分析 - LHF, Surface Wind, Humidity Difference
📅 开始时间: 2026-02-06 16:21:13
📁 输出目录: /work/mh1498/m301257/composite_data/


In [2]:
# ============================================================================
# 函数定义
# ============================================================================

def load_detected_events(composite_dir=None):
    """加载已保存的事件检测结果"""
    if composite_dir is None:
        composite_dir = COMPOSITE_DIR
    
    events_file = os.path.join(composite_dir, 'detected_events.json')
    
    print("\n" + "="*80)
    print("📂 加载事件检测结果")
    print("="*80)
    print(f"文件: {events_file}\n")
    
    if not os.path.exists(events_file):
        print(f"⚠️  事件文件未找到: {events_file}")
        return None, False
    
    try:
        with open(events_file, 'r') as f:
            all_events = json.load(f)
        
        print("✅ 事件检测结果加载成功!")
        print(f"\n📊 事件统计:")
        for exp, events_info in all_events.items():
            print(f"   {exp}: {events_info['n_events']} 个事件")
        
        return all_events, True
        
    except Exception as e:
        print(f"❌ 加载失败: {e}")
        return None, False


def compute_composite_with_std(data, event_dates, lags):
    """
    计算时间合成分析及其标准差
    
    Parameters:
    -----------
    data : xr.DataArray
        输入数据 (time, lat, lon) 或 (time, lev, lat, lon)
    event_dates : list
        事件时间索引列表
    lags : range or list
        lag天数范围
    
    Returns:
    --------
    composite_mean : xr.DataArray
        合成平均场 (lag, ...)
    composite_std : xr.DataArray
        合成标准差 (lag, ...)
    n_events : int
        有效事件数
    """
    composites = []
    
    for event_idx in event_dates:
        # 检查边界条件
        if event_idx + min(lags) < 0 or event_idx + max(lags) >= len(data.time):
            continue
        
        # 提取该事件的所有lag天数据
        event_composite = []
        for lag in lags:
            event_composite.append(data.isel(time=event_idx + lag))
        
        event_composite = xr.concat(event_composite, dim='lag')
        composites.append(event_composite)
    
    if len(composites) > 0:
        # 沿事件维度堆叠并计算统计量
        all_composites = xr.concat(composites, dim='event')
        composite_mean = all_composites.mean(dim='event')
        composite_std = all_composites.std(dim='event')
        
        composite_mean['lag'] = list(lags)
        composite_std['lag'] = list(lags)
        n_events = len(composites)
    else:
        composite_mean = None
        composite_std = None
        n_events = 0
    
    return composite_mean, composite_std, n_events


def apply_kf_filter(data, var_name='variable'):
    """对数据应用Kelvin波滤波"""
    print(f"\n🎛️  对 {var_name} 应用CCKW滤波器...")
    
    wave_filter = CCKWFilter(
        ds=data.fillna(0),
        wave_name='kelvin',
        units='w/m^2',
        spd=1,
        n_workers=4
    )
    
    filtered_data = wave_filter.process()
    print(f"   ✅ 滤波完成: {filtered_data.shape}")
    
    return filtered_data


print("✅ 函数定义完成")

✅ 函数定义完成


In [3]:
# ============================================================================
# STEP 1: 加载事件检测结果
# ============================================================================

all_events, success = load_detected_events()

if not success:
    raise ValueError("❌ 无法加载事件检测结果！请先运行事件检测脚本。")

# 合成分析参数
COMPOSITE_LAGS = range(-4, 5)  # -4 to +4 days

print(f"\n📊 合成分析配置:")
print(f"   Lag范围: {min(COMPOSITE_LAGS)} 到 {max(COMPOSITE_LAGS)} 天")


📂 加载事件检测结果
文件: ../composite_data/detected_events.json

✅ 事件检测结果加载成功!

📊 事件统计:
   CNTL: 546 个事件
   P4K: 548 个事件
   4CO2: 552 个事件

📊 合成分析配置:
   Lag范围: -4 到 4 天


In [4]:
# ============================================================================
# STEP 2: 加载原始数据
# ============================================================================

print("\n" + "="*80)
print("📂 加载原始数据")
print("="*80)

# 海洋mask
def _ocean(ds):
    fraction = xr.open_dataarray(r'../processed_data/land_mask_2deg.nc')
    return fraction == 0

ocean_mask = _ocean(None)
print("✅ 海洋mask加载完成")

# 1. 潜热通量 (hfls)
print("\n📦 加载潜热通量数据...")
lhf_data = {
    'CNTL': xr.open_dataarray('/work/mh1498/m301257/processed_data/2d_layers/hfls_cntl/hfls_2deg_interp.nc').where(ocean_mask, drop=False),
    'P4K': xr.open_dataarray('/work/mh1498/m301257/processed_data/2d_layers/hfls_p4k/hfls_2deg_interp.nc').where(ocean_mask, drop=False),
    '4CO2': xr.open_dataarray('/work/mh1498/m301257/processed_data/2d_layers/hfls_4co2/hfls_2deg_interp.nc').where(ocean_mask, drop=False)
}

# 清理无穷值
for exp in EXPERIMENTS:
    lhf_data[exp] = xr.where(np.isinf(lhf_data[exp]), np.nan, lhf_data[exp])
    print(f"   {exp}: {lhf_data[exp].shape}")

# 2. 表面风速 (sfcwind)
print("\n🌬️  加载表面风速数据...")
surface_wind = {
    'CNTL': xr.open_dataset('/work/mh1498/m301257/processed_data/2d_layers/sfcwind_cntl/sfcwind_2deg_interp.nc')['sfcwind'].where(ocean_mask, drop=False),
    'P4K': xr.open_dataset('/work/mh1498/m301257/processed_data/2d_layers/sfcwind_p4k/sfcwind_2deg_interp.nc')['sfcwind'].where(ocean_mask, drop=False),
    '4CO2': xr.open_dataset('/work/mh1498/m301257/processed_data/2d_layers/sfcwind_4co2/sfcwind_2deg_interp.nc')['sfcwind'].where(ocean_mask, drop=False)
}

for exp in EXPERIMENTS:
    print(f"   {exp}: {surface_wind[exp].shape}")

# 3. 比湿差 (qs - qa)
print("\n💧 加载比湿差数据...")
diff_qa_qs_file = '/work/mh1498/m301257/3D_data/difference_qs_qa.nc'
diff_qa_qs_ds = xr.open_dataset(diff_qa_qs_file)

diff_qa_qs = {
    'CNTL': diff_qa_qs_ds['diff_qs-qa_cntl'].where(ocean_mask, drop=False),
    'P4K': diff_qa_qs_ds['diff_qs-qa_p4k'].where(ocean_mask, drop=False),
    '4CO2': diff_qa_qs_ds['diff_qs-qa_4co2'].where(ocean_mask, drop=False)
}

for exp in EXPERIMENTS:
    print(f"   {exp}: {diff_qa_qs[exp].shape}")

print("\n✅ 所有数据加载完成")


📂 加载原始数据
✅ 海洋mask加载完成

📦 加载潜热通量数据...
✅ 海洋mask加载完成

📦 加载潜热通量数据...
   CNTL: (5114, 15, 180)
   P4K: (5114, 15, 180)
   4CO2: (5114, 15, 180)

🌬️  加载表面风速数据...
   CNTL: (5114, 15, 180)
   P4K: (5114, 15, 180)
   4CO2: (5114, 15, 180)

🌬️  加载表面风速数据...
   CNTL: (5114, 15, 180)
   P4K: (5114, 15, 180)
   4CO2: (5114, 15, 180)

💧 加载比湿差数据...
   CNTL: (5114, 15, 180)
   P4K: (5114, 15, 180)
   4CO2: (5114, 15, 180)

💧 加载比湿差数据...
   CNTL: (5114, 15, 180)
   P4K: (5114, 15, 180)
   4CO2: (5114, 15, 180)

✅ 所有数据加载完成
   CNTL: (5114, 15, 180)
   P4K: (5114, 15, 180)
   4CO2: (5114, 15, 180)

✅ 所有数据加载完成


In [5]:
# ============================================================================
# STEP 3: 应用Kelvin波滤波
# ============================================================================

print("\n" + "="*80)
print("🎛️  应用CCKW滤波器")
print("="*80)

# 1. 滤波潜热通量 (注意：乘以-1转换符号)
print("\n📦 滤波潜热通量...")
lhf_kelvin = {}
for exp in EXPERIMENTS:
    lhf_kelvin[exp] = apply_kf_filter(lhf_data[exp] * (-1), var_name=f'LHF_{exp}')

# 2. 滤波表面风速
print("\n🌬️  滤波表面风速...")
surface_wind_kelvin = {}
for exp in EXPERIMENTS:
    surface_wind_kelvin[exp] = apply_kf_filter(surface_wind[exp], var_name=f'SfcWind_{exp}')

# 3. 滤波比湿差
print("\n💧 滤波比湿差...")
diff_qa_qs_kelvin = {}
for exp in EXPERIMENTS:
    diff_qa_qs_kelvin[exp] = apply_kf_filter(diff_qa_qs[exp], var_name=f'DiffQaQs_{exp}')

print("\n✅ 所有变量滤波完成")


🎛️  应用CCKW滤波器

📦 滤波潜热通量...

🎛️  对 LHF_CNTL 应用CCKW滤波器...
🌊 Processing KELVIN wave filter

==================== Loaded Data Information ====================
Type: <class 'xarray.core.dataarray.DataArray'>
Shape: (5114, 15, 180)
Data type: float64
First few values: <xarray.DataArray 'hfls' (time: 5, lat: 15, lon: 180)> Size: 108kB
dask.array<getitem, shape=(5, 15, 180), dtype=float64, chunksize=(5, 15, 180), chunktype=numpy.ndarray>
Coordinates:
  * time     (time) datetime64[ns] 40B 1980-01-01 1980-01-02 ... 1980-01-05
  * lat      (lat) float64 120B -14.0 -12.0 -10.0 -8.0 ... 8.0 10.0 12.0 14.0
  * lon      (lon) float64 1kB 0.0 2.0 4.0 6.0 8.0 ... 352.0 354.0 356.0 358.0
⏳ Detrending data...
⏳ Performing FFT...
⏳ Applying filter...
⏳ Performing inverse FFT...
⏳ Creating output...

==================== Loaded Data Information ====================
Type: <class 'xarray.core.dataarray.DataArray'>
Shape: (5114, 15, 180)
Data type: float64
First few values: <xarray.DataArray 'hfls' (time: 5

In [6]:
# ============================================================================
# STEP 4: 时间合成分析
# ============================================================================

print("\n" + "="*80)
print("📊 执行时间合成分析")
print("="*80)

# 存储所有合成结果
composites = {
    'lhf': {},
    'surface_wind': {},
    'diff_qa_qs': {}
}

for exp in EXPERIMENTS:
    print(f"\n🔄 处理实验: {exp}")
    print("-" * 70)
    
    event_dates = all_events[exp]['event_dates']
    n_events = all_events[exp]['n_events']
    print(f"   使用 {n_events} 个事件")
    
    # 1. 潜热通量合成
    print(f"\n   📦 潜热通量合成...")
    lhf_composite, lhf_std, n_valid = compute_composite_with_std(
        data=lhf_kelvin[exp],
        event_dates=event_dates,
        lags=COMPOSITE_LAGS
    )
    composites['lhf'][exp] = {
        'composite': lhf_composite,
        'std': lhf_std,
        'n_events': n_valid
    }
    print(f"      ✅ 有效事件: {n_valid}, Shape: {lhf_composite.shape}")
    
    # 2. 表面风速合成
    print(f"\n   🌬️  表面风速合成...")
    wind_composite, wind_std, n_valid = compute_composite_with_std(
        data=surface_wind_kelvin[exp],
        event_dates=event_dates,
        lags=COMPOSITE_LAGS
    )
    composites['surface_wind'][exp] = {
        'composite': wind_composite,
        'std': wind_std,
        'n_events': n_valid
    }
    print(f"      ✅ 有效事件: {n_valid}, Shape: {wind_composite.shape}")
    
    # 3. 比湿差合成
    print(f"\n   💧 比湿差合成...")
    qa_qs_composite, qa_qs_std, n_valid = compute_composite_with_std(
        data=diff_qa_qs_kelvin[exp],
        event_dates=event_dates,
        lags=COMPOSITE_LAGS
    )
    composites['diff_qa_qs'][exp] = {
        'composite': qa_qs_composite,
        'std': qa_qs_std,
        'n_events': n_valid
    }
    print(f"      ✅ 有效事件: {n_valid}, Shape: {qa_qs_composite.shape}")

print("\n✅ 所有合成分析完成")


📊 执行时间合成分析

🔄 处理实验: CNTL
----------------------------------------------------------------------
   使用 546 个事件

   📦 潜热通量合成...
      ✅ 有效事件: 546, Shape: (9, 15, 180)

   🌬️  表面风速合成...
      ✅ 有效事件: 546, Shape: (9, 15, 180)

   🌬️  表面风速合成...
      ✅ 有效事件: 546, Shape: (9, 15, 180)

   💧 比湿差合成...
      ✅ 有效事件: 546, Shape: (9, 15, 180)

   💧 比湿差合成...
      ✅ 有效事件: 546, Shape: (9, 15, 180)

🔄 处理实验: P4K
----------------------------------------------------------------------
   使用 548 个事件

   📦 潜热通量合成...
      ✅ 有效事件: 546, Shape: (9, 15, 180)

🔄 处理实验: P4K
----------------------------------------------------------------------
   使用 548 个事件

   📦 潜热通量合成...
      ✅ 有效事件: 548, Shape: (9, 15, 180)

   🌬️  表面风速合成...
      ✅ 有效事件: 548, Shape: (9, 15, 180)

   🌬️  表面风速合成...
      ✅ 有效事件: 548, Shape: (9, 15, 180)

   💧 比湿差合成...
      ✅ 有效事件: 548, Shape: (9, 15, 180)

   💧 比湿差合成...
      ✅ 有效事件: 548, Shape: (9, 15, 180)

🔄 处理实验: 4CO2
----------------------------------------------------------------------

In [7]:
# ============================================================================
# STEP 5: 保存合成结果
# ============================================================================

print("\n" + "="*80)
print("💾 保存合成结果到文件")
print("="*80)
print(f"📁 输出目录: {OUTPUT_DIR}\n")

for exp in EXPERIMENTS:
    exp_lower = exp.lower()
    print(f"🔄 保存 {exp} 数据...")
    
    # 1. 保存潜热通量合成
    lhf_file = os.path.join(OUTPUT_DIR, f'lhf_composite_{exp_lower}.nc')
    composites['lhf'][exp]['composite'].to_netcdf(lhf_file)
    file_size = os.path.getsize(lhf_file) / 1024
    print(f"   ✅ 潜热通量: {os.path.basename(lhf_file)} ({file_size:.1f} KB)")
    
    # 2. 保存表面风速合成
    wind_file = os.path.join(OUTPUT_DIR, f'surface_wind_composite_{exp_lower}.nc')
    composites['surface_wind'][exp]['composite'].to_netcdf(wind_file)
    file_size = os.path.getsize(wind_file) / 1024
    print(f"   ✅ 表面风速: {os.path.basename(wind_file)} ({file_size:.1f} KB)")
    
    # 3. 保存比湿差合成
    qa_qs_file = os.path.join(OUTPUT_DIR, f'diff_qa_qs_composite_{exp_lower}.nc')
    composites['diff_qa_qs'][exp]['composite'].to_netcdf(qa_qs_file)
    file_size = os.path.getsize(qa_qs_file) / 1024
    print(f"   ✅ 比湿差: {os.path.basename(qa_qs_file)} ({file_size:.1f} KB)")
    
    print()

print("="*80)
print("✅ 所有数据保存完成!")
print("="*80)
print(f"\n📊 数据摘要:")
print(f"   - 实验数量: {len(EXPERIMENTS)}")
print(f"   - 变量数量: 3 (潜热通量、表面风速、比湿差)")
print(f"   - Lag范围: {min(COMPOSITE_LAGS)} 到 {max(COMPOSITE_LAGS)} 天")
print(f"   - 总文件数: {len(EXPERIMENTS) * 3}")
print(f"\n📅 完成时间: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print("="*80)


💾 保存合成结果到文件
📁 输出目录: /work/mh1498/m301257/composite_data/

🔄 保存 CNTL 数据...
   ✅ 潜热通量: lhf_composite_cntl.nc (199.4 KB)
   ✅ 表面风速: surface_wind_composite_cntl.nc (199.4 KB)
   ✅ 比湿差: diff_qa_qs_composite_cntl.nc (199.4 KB)

🔄 保存 P4K 数据...
   ✅ 潜热通量: lhf_composite_p4k.nc (199.4 KB)
   ✅ 表面风速: surface_wind_composite_p4k.nc (199.4 KB)
   ✅ 比湿差: diff_qa_qs_composite_p4k.nc (199.4 KB)

🔄 保存 4CO2 数据...
   ✅ 潜热通量: lhf_composite_4co2.nc (199.4 KB)
   ✅ 表面风速: surface_wind_composite_4co2.nc (199.4 KB)
   ✅ 比湿差: diff_qa_qs_composite_4co2.nc (199.4 KB)

✅ 所有数据保存完成!

📊 数据摘要:
   - 实验数量: 3
   - 变量数量: 3 (潜热通量、表面风速、比湿差)
   - Lag范围: -4 到 4 天
   - 总文件数: 9

📅 完成时间: 2026-02-06 16:22:15
